# Crop Recommendation System - Data Science & Machine Learning Notebook

This notebook contains the complete workflow for the **Crop Recommendation System** project. We will load the dataset, perform basic exploratory data analysis (EDA), visualize the relationship between environmental factors and crops, pre-process the data, train **Decision Tree** and **Random Forest** classification models, evaluate their performance, and extract feature importances.

### Features present in the dataset:
1. **N**: Ratio of Nitrogen content in soil
2. **P**: Ratio of Phosphorous content in soil
3. **K**: Ratio of Potassium content in soil
4. **temperature**: Temperature in degrees Celsius
5. **humidity**: Relative humidity in %
6. **ph**: pH value of the soil (acidity/alkalinity)
7. **rainfall**: Rainfall in mm
8. **label**: Crop type (Target Variable)

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

print("Libraries imported successfully.")

## 1. Loading the Dataset

In [ ]:
# Load the dataset
data_path = "../data/Crop_recommendation.csv"

if os.path.exists(data_path):
    df = pd.read_csv(data_path)
    print(f"Dataset loaded. Shape: {df.shape}")
else:
    # Fallback to download from URL directly in notebook
    url = "https://raw.githubusercontent.com/arzzahid66/Optimizing_Agricultural_Production/master/Crop_recommendation.csv"
    df = pd.read_csv(url)
    print(f"Dataset loaded from URL. Shape: {df.shape}")

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# Display first few rows
df.head()

In [ ]:
# Display statistical summary
df.describe()

In [ ]:
# Check for missing values
print("Missing values:")
print(df.isnull().sum())

In [ ]:
# List unique crops
unique_crops = df['label'].unique()
print(f"Total unique crops: {len(unique_crops)}")
print(unique_crops)

## 3. Data Visualizations

In [ ]:
# 3.1 Class distribution count
plt.figure(figsize=(12, 6))
df['label'].value_counts().plot(kind='bar', color='teal')
plt.title('Distribution of Crops in Dataset')
plt.xlabel('Crop Name')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# 3.2 Correlation heatmap among numeric features
plt.figure(figsize=(8, 6))
corr_matrix = df.drop('label', axis=1).corr()
plt.imshow(corr_matrix, cmap='coolwarm', interpolation='nearest')
plt.colorbar(label='Correlation Coefficient')
plt.xticks(range(len(corr_matrix.columns)), corr_matrix.columns, rotation=45)
plt.yticks(range(len(corr_matrix.columns)), corr_matrix.columns)
plt.title('Correlation Matrix of Soil and Climatic Features')
for i in range(len(corr_matrix.columns)):
    for j in range(len(corr_matrix.columns)):
        plt.text(j, i, f"{corr_matrix.iloc[i, j]:.2f}", ha='center', va='center', color='black')
plt.tight_layout()
plt.show()

## 4. Model Training & Preprocessing

In [ ]:
# Define independent and dependent variables
X = df.drop('label', axis=1)
y = df['label']

# Split dataset (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

### 4.1 Decision Tree Classifier

In [ ]:
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)

dt_predictions = dt_model.predict(X_test)
dt_accuracy = accuracy_score(y_test, dt_predictions)
print(f"Decision Tree Test Accuracy: {dt_accuracy:.4f}")
print("\nDecision Tree Classification Report:")
print(classification_report(y_test, dt_predictions))

### 4.2 Random Forest Classifier

In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

rf_predictions = rf_model.predict(X_test)
rf_accuracy = accuracy_score(y_test, rf_predictions)
print(f"Random Forest Test Accuracy: {rf_accuracy:.4f}")
print("\nRandom Forest Classification Report:")
print(classification_report(y_test, rf_predictions))

## 5. Model Comparison & Analysis

In [ ]:
# Accuracy Comparison plot
models = ['Decision Tree', 'Random Forest']
accuracies = [dt_accuracy * 100, rf_accuracy * 100]

plt.figure(figsize=(6, 4))
bars = plt.bar(models, accuracies, color=['coral', 'emerald'])
plt.ylabel('Accuracy (%)')
plt.ylim(90, 100)
plt.title('Classifier Accuracy Comparison')
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 0.2, f"{yval:.2f}%", ha='center', va='bottom')
plt.show()

### 5.1 Feature Importance (Random Forest)

In [ ]:
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1]
features = X.columns

plt.figure(figsize=(10, 6))
plt.title('Feature Importances in Random Forest')
plt.bar(range(X.shape[1]), importances[indices], color='mediumseagreen', align='center')
plt.xticks(range(X.shape[1]), [features[i] for i in indices], rotation=45)
plt.xlim([-1, X.shape[1]])
plt.ylabel('Importance Score')
plt.tight_layout()
plt.show()

## 6. Testing Sample Prediction

In [ ]:
# Example input from problem statement:
# N = 90, P = 42, K = 43, Temperature = 21, Humidity = 82, pH = 6.5, Rainfall = 203
sample_input = np.array([[90, 42, 43, 21, 82, 6.5, 203]])
predicted_crop = rf_model.predict(sample_input)[0]
predicted_prob = rf_model.predict_proba(sample_input)[0]

print(f"Predicted Crop: {predicted_crop}")
print(f"Confidence Score: {np.max(predicted_prob) * 100:.2f}%")